In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from plasmapy.diagnostics import thomson

from forward import spectral_density
from scipy.constants import c, k as kB, epsilon_0, e, m_p

from dispersion import _Zprime
import time

# Forward Model Validation

First let's validate the forward model, comparing plasmapy.diagnostics.thomson.spectral_density to forward.spectral_density. We first generate some sample time-resolved profiles of the relevant plasma parameters:

In [ ]:
Nt = 51 #number of timesteps
t = np.linspace(0, 3, Nt) #ns time
tau = 1.5 #time over which plasma parameters vary

#Plasma parameters
ne = 1e20 * np.exp(-t / tau)
ue = 1e6 * np.exp(-np.sqrt(t / tau))
ui = ue + 2e5 * np.exp(-t / tau) - 1e5
Te = 1000 * np.exp(-t / tau)
Ti = 2000 - np.sqrt(t) * 1000
ifractC = 0.05 * t**2 #deuterons
ifractD = 1 - ifractC #carbon

nD = ne * ifractD
nC = ne * ifractC / 6

#plasmapy defines ifract differently...
ifractD_old = nD / (nD + nC)
ifractC_old = nC / (nD + nC)



#Plot ground truth profiles
fig, ax = plt.subplots(nrows=3, figsize=(4, 9))

ax[0].semilogy(t, ne, label = "Electrons")
ax[0].plot(t, nD, label = "Deuterons")
ax[0].plot(t, nC, label = "Carbon ions")
ax[0].set_title("Densities")
ax[0].set_ylabel("n [cm^-3]")
ax[0].legend()

ax[1].plot(t, ue, label = "Electrons")
ax[1].plot(t, ui, label = "Ions")
ax[1].set_title("Flow velocities")
ax[1].set_ylabel("u [m / s]")
ax[1].legend()

ax[2].plot(t, Te, label = "Electrons")
ax[2].plot(t, Ti, label = "Ions")
ax[2].set_title("Temperatures")
ax[2].set_ylabel("T [eV]")
ax[2].legend()

ax[2].set_xlabel("Time [ns]")

fig.tight_layout()

Next we define the parameters of the Thomson diagnostic for both the EPW and IAW measurements

In [ ]:
def sph2cart(phi, theta):
    phi = phi / 180 * np.pi
    theta = theta / 180 * np.pi
    ans = np.array([np.sin(phi) * np.cos(theta), np.sin(phi) * np.sin(theta), np.cos(phi)])
    return ans / np.linalg.norm(ans)

In [ ]:
probe_wavelength = 263.25 #4w on OMEGA
epw_lam = np.linspace(probe_wavelength - 30, probe_wavelength + 30, 1024)#
#epw_lam = np.linspace(235 - 20, 235 + 20, 1024) #Only measuring the blue peak
iaw_lam = np.linspace(probe_wavelength - 2, probe_wavelength + 2, 1024)


probe_vec = sph2cart(116.57, 18)
scatter_vec = -sph2cart(112.15, 162)
k_vec = scatter_vec - probe_vec
k_vec = k_vec / np.linalg.norm(k_vec)

Generate the Skw over all times using plasmapy + loop vs. my code

In [ ]:
e / kB

In [ ]:
epw_Skw_plasmapy = np.zeros((len(epw_lam), Nt))
iaw_Skw_plasmapy = np.zeros((len(iaw_lam), Nt))

t0 = time.time()
for i in range(Nt):
    epw_Skw_plasmapy[:, i] = thomson.spectral_density_lite(
        wavelengths=epw_lam * 1e-9,
        probe_wavelength=probe_wavelength * 1e-9,
        n = ne[i] * 1e6,
        T_e = np.array([Te[i]]) * e / kB,
        T_i = np.array([Ti[i], Ti[i]]) * e / kB,
        efract = np.array([1]),
        ifract = np.array([ifractD_old[i], ifractC_old[i]]),
        ion_z = np.array([1, 6]),
        ion_mass = np.array([2, 12]) * m_p,
        electron_vel = np.array([k_vec * ue[i]]),
        ion_vel = np.array([k_vec * ui[i], k_vec * ui[i]]),
        probe_vec=probe_vec,
        scatter_vec=scatter_vec,
    )[1]
    iaw_Skw_plasmapy[:, i] = thomson.spectral_density_lite(
        wavelengths=iaw_lam * 1e-9,
        probe_wavelength=probe_wavelength * 1e-9,
        n = ne[i] * 1e6,
        T_e = np.array([Te[i]])[:, ] * e / kB,
        T_i = np.array([Ti[i], Ti[i]]) * e / kB,
        efract = np.array([1]),
        ifract = np.array([ifractD_old[i], ifractC_old[i]]),
        ion_z = np.array([1, 6]),
        ion_mass = np.array([2, 12]) * m_p,
        electron_vel = np.array([k_vec * ue[i]]),
        ion_vel = np.array([k_vec * ui[i], k_vec * ui[i]]),
        probe_vec=probe_vec,
        scatter_vec=scatter_vec,
    )[1]
t1 = time.time()
dt = t1 - t0
print("PlasmaPy runtime:", dt)

t0 = time.time()
epw_Skw_new = spectral_density(
    n = ne*1e6,
    Te = np.array([Te]) * e / kB,
    Ti = np.array([Ti, Ti]) * e / kB,
    ue = np.array([ue]),
    ui = np.array([ui, ui]),
    pe = np.ones_like(ne) * 2,
    pi = np.ones_like(np.array([ui, ui])) * 2,
    efract = np.array([np.ones_like(ne)]),
    ifract = np.array([ifractD, ifractC]),
    ion_z = np.array([1, 6]),
    ion_a = np.array([2, 12]),
    wavelengths=epw_lam * 1e-9,
    probe_wavelength=probe_wavelength * 1e-9,
    probe_vec=probe_vec,
    scatter_vec=scatter_vec,
    ue_dir=k_vec,
    ui_dir=k_vec,
)

iaw_Skw_new = spectral_density(
    n = ne * 1e6,
    Te = np.array([Te]) * e / kB,
    Ti = np.array([Ti, Ti]) * e / kB,
    ue = np.array([ue]),
    ui = np.array([ui, ui]),
    pe = np.ones_like(ne) * 2,
    pi = np.ones_like(np.array([ui, ui])) * 2,
    efract = np.array([np.ones_like(ne)]),
    ifract = np.array([ifractD, ifractC]),
    ion_z = np.array([1, 6]),
    ion_a = np.array([2, 12]),
    wavelengths=iaw_lam * 1e-9,
    probe_wavelength=probe_wavelength * 1e-9,
    probe_vec=probe_vec,
    scatter_vec=scatter_vec,
    ue_dir=k_vec,
    ui_dir=k_vec,
)

t1 = time.time()
dt = t1 - t0
print("New time:", dt)

#plt.ylim(-1, 0)

In [ ]:
fig, ax = plt.subplots(nrows = 2, ncols = 2, figsize = (10, 10))

ax[0,0].pcolormesh(t, epw_lam, epw_Skw_plasmapy)
ax[0,1].pcolormesh(t, iaw_lam, iaw_Skw_plasmapy)
ax[0, 0].set_title("PlasmaPy - EPW Spectrum")
ax[0, 1].set_title("PlasmaPy - IAW Spectrum")

ax[1,0].pcolormesh(t, epw_lam, epw_Skw_new)
ax[1,1].pcolormesh(t, iaw_lam, iaw_Skw_new)
ax[1, 0].set_title("New - EPW Spectrum")
ax[1, 1].set_title("New - IAW Spectrum")

ax[1, 0].set_xlabel("Time [ns]")
ax[1, 1].set_xlabel("Time [ns]")
ax[0, 0].set_ylabel("Wavelength [nm]")
ax[1, 0].set_ylabel("Wavelength [nm]")
#ax[1, 1].set_ylim(260, 265)

In [ ]:
fig, ax = plt.subplots(nrows = 2, figsize = (6, 8))

ax[0].plot(epw_lam, epw_Skw_plasmapy[:, 50])
ax[0].set_ylim(0, 8e-15)
ax[0].set_ylabel(r"Spectral density")
ax[0].set_title("Electron Plasma Wave (EPW) Feature", fontsize=14)
ax[0].vlines([261.25, 265.25], 0, 8e-15, color = 'k', linestyle = '--')

ax[1].plot(iaw_lam, iaw_Skw_plasmapy[:, 50])
ax[1].set_ylabel(r"Spectral density")
ax[1].set_xlabel("Wavelength [nm]")
ax[1].set_title("Ion Acoustic Wave (IAW) Feature", fontsize=14)
fig.suptitle("Synthetic collective Thomson scattering data", fontsize = 20)

plt.savefig("synthetic_thomson.png", dpi = 300, transparent = True, bbox_inches = "tight")